# Phase-level Metrics and Visualization

This notebook compares primary-adapted RMSE and MAE during take-off,
cruise, and landing for the seven CMU target cells.

DTNN phase metrics are averaged across the five formal repeats from
`phase_metrics_full_mission_diagnostic.csv`. TATN phase metrics are read
from the verified five-repeat `phase_metrics_summary.csv`. Both therefore
use the same five-repeat reporting convention as the cell-level tables.
The DTNN phase diagnostic covers complete held-out missions, while its
primary overall metric retains the original initial-SOC-70 evaluation
segment.


In [ ]:
"""Code Cell 1 -- imports, Drive mount, and Drive paths.

Same hardcoded-Drive-path convention as notebooks 37 and 38: no dependency
on a cloned repo checkout inside the Colab runtime, only that Drive is
mounted."""

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

try:
    import google.colab  # noqa: F401

    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

DRIVE_ROOT = Path("/content/drive/MyDrive/battery_soc_benchmark")
OUTPUT_ROOT = DRIVE_ROOT / "outputs"
BENCH_ROOT = OUTPUT_ROOT / "benchmarks" / "panasonic_to_cmu_evtol"

print("DRIVE_ROOT  :", DRIVE_ROOT)
print("OUTPUT_ROOT :", OUTPUT_ROOT)
print("BENCH_ROOT  :", BENCH_ROOT)


In [ ]:
"""Code Cell 2 -- configuration: cells, phase order, per-framework paths."""

# Same fixed cell order as notebooks 37/38.
CELL_IDS = ["VAH01", "VAH17", "VAH27", "VAH30", "VAH15", "VAH12", "VAH11"]

# Fixed globally, verified against real phase-metric files (see title-cell
# note 1/2) -- never auto-derived per cell.
PHASE_ORDER = ["take_off", "cruise", "landing"]
PHASE_DISPLAY_MAP = {"take_off": "Take-off", "cruise": "Cruise", "landing": "Landing"}

# Frameworks with verified phase-level metrics.
FRAMEWORK_ROW_ORDER = [
    "Vidal Fine-tuning",
    "CNN-LSTM",
    "TCN",
    "Transformer Encoder",
    "DTNN",
    "TATN",
    "DARM",
    "SFTNN (MED)",
]

# Same selection rule as notebook 38's PRIMARY_ADAPTED_SELECTOR, redefined
# identically here (no cross-notebook import dependency).
PRIMARY_ADAPTED_SELECTOR = {
    "Vidal Fine-tuning": ("budget_internal", "vidal_full"),
    "CNN-LSTM": ("budget_internal", "full"),
    "TCN": ("budget_internal", "full"),
    "Transformer Encoder": ("budget_internal", "full"),
    "DTNN": ("method", "Bian MDA"),
    "TATN": ("method", "TATN"),
    "DARM": ("method", "DARM"),
    "SFTNN (MED)": ("method", "med"),
}

# ---- Vidal: only the vidal_full setting is needed (Primary adapted only). ----
VIDAL_ROOT = BENCH_ROOT / "vidal"
VIDAL_SEED = 42
VIDAL_FULL_RUN_ID = "20260916_192416"
VIDAL_FULL_METRICS = VIDAL_ROOT / "vidal_full" / f"seed_{VIDAL_SEED}" / VIDAL_FULL_RUN_ID / "cell_metrics.csv"

# ---- Backbone: one shared phase_metrics.csv, split by `model`. ----
BACKBONE_DIR = BENCH_ROOT / "backbone_transfer_ns_relabel_v2"
BACKBONE_PHASE_METRICS = BACKBONE_DIR / "phase_metrics.csv"
BACKBONE_MODEL_TO_DISPLAY_NAME = {
    "cnn_lstm": "CNN-LSTM",
    "tcn": "TCN",
    "transformer_encoder": "Transformer Encoder",
}

# ---- DTNN: repeat-level full-mission phase diagnostics, averaged over
# the five formal repeats for table/figure reporting. ----
DTNN_DIR = BENCH_ROOT / "bian_2021" / "initial_soc_70_v2"
DTNN_FORMAL_PHASE_METRICS = DTNN_DIR / "formal" / "phase_metrics_full_mission_diagnostic.csv"
DTNN_PHASE_REPEATS = 5

# ---- DARM: confirmed-current v2 folder (NOT the older v1 decoy). ----
DARM_DIR = BENCH_ROOT / "liu_2025_darm_ns_relabel_v2_e30_clean"
DARM_PHASE_METRICS = DARM_DIR / "phase_metrics.csv"

# ---- SFTNN. ----
SFTNN_DIR = BENCH_ROOT / "sftnn_med_transfer_ns_relabel_v2"
SFTNN_PHASE_METRICS = SFTNN_DIR / "phase_metrics.csv"

# ---- TATN: verified current-protocol five-repeat phase summary. ----
TATN_DIR = BENCH_ROOT / "tatn_ns_relabel_v2_unsupervised_formal"
TATN_PHASE_SUMMARY = TATN_DIR / "phase_metrics_summary.csv"

OUTPUT_DIR = Path("phase_metric_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
FIGURES_DIR = OUTPUT_DIR  # per-cell PNGs are written directly under OUTPUT_DIR per the requested layout

print("CELL_IDS:", CELL_IDS)
print("PHASE_ORDER:", PHASE_ORDER)
print("PHASE_DISPLAY_MAP:", PHASE_DISPLAY_MAP)
print("FRAMEWORK_ROW_ORDER:", FRAMEWORK_ROW_ORDER)


In [ ]:
"""Code Cell -- real-Drive path existence diagnostics for every phase-level
metric source this notebook reads, before any loader runs."""

_diagnostic_paths = {
    "VIDAL_FULL_METRICS": VIDAL_FULL_METRICS,
    "BACKBONE_PHASE_METRICS": BACKBONE_PHASE_METRICS,
    "DTNN_FORMAL_PHASE_METRICS": DTNN_FORMAL_PHASE_METRICS,
    "DARM_PHASE_METRICS": DARM_PHASE_METRICS,
    "SFTNN_PHASE_METRICS": SFTNN_PHASE_METRICS,
    "TATN_PHASE_SUMMARY": TATN_PHASE_SUMMARY,
}

print("=" * 78)
print("PATH EXISTENCE DIAGNOSTICS (real Drive, checked before loading)")
print("=" * 78)
_any_missing = False
for name, p in _diagnostic_paths.items():
    exists = p.exists()
    _any_missing = _any_missing or not exists
    print(f"{name:30s} {str(p.resolve()) if exists else str(p):100s} exists={exists}")
if _any_missing:
    print("\n[WARNING] One or more expected paths are missing on this Drive mount. "
          "The corresponding loader below will emit a skip warning and that framework will "
          "be left out of the affected cell's table/plot rather than guessed.")
else:
    print("\nAll checked paths exist.")


In [ ]:
"""Code Cell 3 -- metric-unit normalization helper (same heuristic as
notebook 38's ensure_percent_scale)."""


def ensure_percent_scale(series: pd.Series) -> pd.Series:
    if series.empty:
        return series
    s = pd.to_numeric(series, errors="coerce")
    if s.dropna().empty:
        return s
    if s.abs().max() <= 1.5:
        return s * 100.0
    return s


In [ ]:
"""Code Cell 4 -- per-framework phase-level metric loaders.

Each loader returns a list of dict records with the common
`phase_metric_long` schema: cell_id, framework, phase_internal,
phase_display, rmse, mae, evaluation_unit, source_file, source_path,
method, model, budget, seed_or_repeat, is_screening,
formal_benchmark_eligible, notes.

Every loader prefers an existing recorded phase-metrics file over
recomputing from predictions; none of these frameworks required falling
back to prediction-level computation (all had a real phase_metrics.csv, or
-- for Vidal -- embedded phase columns in cell_metrics.csv)."""

_skip_warnings = []


def _read_csv_safely(path: Path) -> pd.DataFrame | None:
    if not path.exists():
        return None
    return pd.read_csv(path)


def load_vidal_phase_records() -> list[dict]:
    records = []
    df = _read_csv_safely(VIDAL_FULL_METRICS)
    if df is None:
        _skip_warnings.append("Vidal Fine-tuning: vidal_full cell_metrics.csv not found -- skipped entirely.")
        return records
    for _, row in df.iterrows():
        for phase in PHASE_ORDER:
            rmse_col, mae_col = f"{phase}_rmse", f"{phase}_mae"
            if rmse_col not in row or mae_col not in row:
                continue
            rmse = ensure_percent_scale(pd.Series([row[rmse_col]])).iloc[0]
            mae = ensure_percent_scale(pd.Series([row[mae_col]])).iloc[0]
            records.append(dict(
                cell_id=row["cell_id"], framework="Vidal Fine-tuning",
                phase_internal=phase, phase_display=PHASE_DISPLAY_MAP[phase],
                rmse=rmse, mae=mae, evaluation_unit="last_step_per_window",
                source_file="cell_metrics.csv", source_path=str(VIDAL_FULL_METRICS),
                method="fine_tuned", model=None, budget="vidal_full",
                seed_or_repeat=VIDAL_SEED, is_screening=False, formal_benchmark_eligible=False,
                notes="Per-phase columns embedded directly in Vidal's cell_metrics.csv "
                      "(no separate phase_metrics.csv exists for Vidal); no per-phase "
                      "point count is recorded in this file (only overall test_windows), "
                      "so n_points/n_missions are left blank for Vidal rows rather than fabricated.",
            ))
    return records


def load_backbone_phase_records() -> list[dict]:
    records = []
    df = _read_csv_safely(BACKBONE_PHASE_METRICS)
    if df is None:
        for display in BACKBONE_MODEL_TO_DISPLAY_NAME.values():
            _skip_warnings.append(f"{display}: phase_metrics.csv not found -- skipped entirely.")
        return records
    full_only = df[df["budget"] == "full"]
    for _, row in full_only.iterrows():
        model = row["model"]
        display = BACKBONE_MODEL_TO_DISPLAY_NAME.get(model, model)
        phase = row["mission_phase"]
        if phase not in PHASE_ORDER:
            continue
        records.append(dict(
            cell_id=row["cell_id"], framework=display,
            phase_internal=phase, phase_display=PHASE_DISPLAY_MAP[phase],
            rmse=row.get("rmse_percent", np.nan), mae=row.get("mae_percent", np.nan),
            evaluation_unit="all_timesteps", source_file="phase_metrics.csv",
            source_path=str(BACKBONE_PHASE_METRICS), method="fine_tuned", model=model,
            budget="full", seed_or_repeat=None,
            n_points=row.get("n_points", np.nan), n_missions=row.get("n_missions", np.nan),
            is_screening=False, formal_benchmark_eligible=False, notes="",
        ))
    return records


def load_dtnn_phase_records() -> list[dict]:
    records = []
    df = _read_csv_safely(DTNN_FORMAL_PHASE_METRICS)
    if df is None:
        _skip_warnings.append(
            "DTNN: formal phase_metrics_full_mission_diagnostic.csv not found -- skipped."
        )
        return records

    mda = df[df["method"] == "Bian MDA"].copy()
    grouped = (
        mda.groupby(["cell_id", "mission_phase"], as_index=False)
        .agg(
            rmse=("rmse_percent", "mean"),
            mae=("mae_percent", "mean"),
            n_points=("n_points", "mean"),
            n_missions=("n_missions", "mean"),
            n_repeats=("repeat", "nunique"),
        )
    )
    for _, row in grouped.iterrows():
        phase = row["mission_phase"]
        if phase not in PHASE_ORDER:
            continue
        records.append(dict(
            cell_id=row["cell_id"], framework="DTNN",
            phase_internal=phase, phase_display=PHASE_DISPLAY_MAP[phase],
            rmse=row["rmse"], mae=row["mae"], evaluation_unit="all_timesteps",
            source_file="phase_metrics_full_mission_diagnostic.csv",
            source_path=str(DTNN_FORMAL_PHASE_METRICS), method="Bian MDA",
            model=None, budget=None, seed_or_repeat="mean_of_5",
            n_points=row["n_points"], n_missions=row["n_missions"],
            is_screening=False, formal_benchmark_eligible=False,
            notes=(
                f"Mean across {int(row['n_repeats'])} formal repeats; "
                "complete held-out-mission diagnostic."
            ),
        ))
    return records


def load_darm_phase_records() -> list[dict]:
    records = []
    df = _read_csv_safely(DARM_PHASE_METRICS)
    if df is None:
        _skip_warnings.append("DARM: phase_metrics.csv not found -- skipped entirely.")
        return records
    darm_only = df[df["method"] == "DARM"]
    for _, row in darm_only.iterrows():
        phase = row["mission_phase"]
        if phase not in PHASE_ORDER:
            continue
        records.append(dict(
            cell_id=row["cell_id"], framework="DARM",
            phase_internal=phase, phase_display=PHASE_DISPLAY_MAP[phase],
            rmse=row.get("rmse_percent", np.nan), mae=row.get("mae_percent", np.nan),
            evaluation_unit="last_step_per_window", source_file="phase_metrics.csv",
            source_path=str(DARM_PHASE_METRICS), method="DARM", model=None,
            budget=None, seed_or_repeat=None,
            n_points=row.get("n_points", np.nan), n_missions=row.get("n_missions", np.nan),
            is_screening=False, formal_benchmark_eligible=False,
            notes=f"condition={row.get('condition')!r} (DARM's per-cell OOD test condition, "
                  "not a budget tier).",
        ))
    return records


def load_sftnn_phase_records() -> list[dict]:
    records = []
    df = _read_csv_safely(SFTNN_PHASE_METRICS)
    if df is None:
        _skip_warnings.append("SFTNN (MED): phase_metrics.csv not found -- skipped entirely.")
        return records
    med_only = df[df["method"] == "med"]
    for _, row in med_only.iterrows():
        phase = row["mission_phase"]
        if phase not in PHASE_ORDER:
            continue
        records.append(dict(
            cell_id=row["cell_id"], framework="SFTNN (MED)",
            phase_internal=phase, phase_display=PHASE_DISPLAY_MAP[phase],
            rmse=row.get("rmse_percent", np.nan), mae=row.get("mae_percent", np.nan),
            evaluation_unit="all_timesteps", source_file="phase_metrics.csv",
            source_path=str(SFTNN_PHASE_METRICS), method="med", model=None,
            budget=None, seed_or_repeat=None,
            n_points=row.get("n_points", np.nan), n_missions=row.get("n_missions", np.nan),
            is_screening=False, formal_benchmark_eligible=False,
            notes="SFTNN MED uses no target labels at all during transfer (pseudo-labels only).",
        ))
    return records


def load_tatn_phase_records() -> list[dict]:
    records = []
    df = _read_csv_safely(TATN_PHASE_SUMMARY)
    if df is None:
        _skip_warnings.append("TATN: phase_metrics_summary.csv not found -- skipped.")
        return records

    adapted = df[
        (df["scope"] == "cell_across_repeats")
        & (df["method"] == "TATN")
        & df["cell_id"].notna()
    ]
    for _, row in adapted.iterrows():
        phase = row["mission_phase"]
        if phase not in PHASE_ORDER:
            continue
        records.append(dict(
            cell_id=row["cell_id"], framework="TATN",
            phase_internal=phase, phase_display=PHASE_DISPLAY_MAP[phase],
            rmse=row.get("rmse_percent_mean", np.nan),
            mae=row.get("mae_percent_mean", np.nan),
            evaluation_unit="all_timesteps",
            source_file="phase_metrics_summary.csv",
            source_path=str(TATN_PHASE_SUMMARY), method="TATN",
            model=None, budget=None, seed_or_repeat="mean_of_5",
            n_points=row.get("n_points_mean", np.nan),
            n_missions=row.get("n_missions_mean", np.nan),
            is_screening=False, formal_benchmark_eligible=True,
            notes=f"Mean across {int(row.get('n', 5))} formal repeats.",
        ))
    return records


In [ ]:
"""Code Cell 5 -- build the long-form master table `phase_metric_long`."""

_all_phase_records = []
_all_phase_records += load_vidal_phase_records()
_all_phase_records += load_backbone_phase_records()
_all_phase_records += load_dtnn_phase_records()
_all_phase_records += load_darm_phase_records()
_all_phase_records += load_sftnn_phase_records()
_all_phase_records += load_tatn_phase_records()

phase_metric_long = pd.DataFrame.from_records(_all_phase_records)

_expected_cols = [
    "cell_id", "framework", "phase_internal", "phase_display", "rmse", "mae",
    "evaluation_unit", "source_file", "source_path", "method", "model", "budget",
    "seed_or_repeat", "n_points", "n_missions", "is_screening",
    "formal_benchmark_eligible", "notes",
]
phase_metric_long = phase_metric_long.reindex(columns=_expected_cols)
phase_metric_long["phase_internal"] = pd.Categorical(
    phase_metric_long["phase_internal"], categories=PHASE_ORDER, ordered=True
)

_n_fw = phase_metric_long["framework"].nunique()
print(f"phase_metric_long: {len(phase_metric_long)} rows, {_n_fw} frameworks")
print(phase_metric_long["framework"].value_counts())
print("\nSkip warnings:")
for w in _skip_warnings:
    print("  [SKIP]", w)
phase_metric_long.head(10)


In [ ]:
"""Code Cell 6 -- pre-plot diagnostics: print every (cell, framework, phase)
combination with its point/mission count, metric values, evaluation_unit
and source; flag zero-point combos, DARM/Vidal sparse coverage, and
phase-label consistency across frameworks."""

print("=" * 100)
print("PHASE-LEVEL DIAGNOSTIC DUMP")
print("=" * 100)
_diag_cols = ["cell_id", "framework", "phase_internal", "n_points", "n_missions",
              "rmse", "mae", "evaluation_unit", "source_file"]
print(phase_metric_long[_diag_cols].sort_values(["framework", "cell_id", "phase_internal"]).to_string(index=False))

print("\n--- Checks ---")
_diag_warnings = []

# Zero-point framework/phase combinations.
_zero_points = phase_metric_long[(phase_metric_long["n_points"] == 0) | (phase_metric_long["n_missions"] == 0)]
if len(_zero_points):
    _diag_warnings.append(f"{len(_zero_points)} rows have n_points==0 or n_missions==0")

# DARM / Vidal sparse-evaluation phase coverage: each should have exactly
# 3 phases per cell (no partial phase coverage).
for framework in ["DARM", "Vidal Fine-tuning"]:
    sub = phase_metric_long[phase_metric_long["framework"] == framework]
    counts = sub.groupby("cell_id", observed=True)["phase_internal"].nunique()
    incomplete = counts[counts < len(PHASE_ORDER)]
    if len(incomplete):
        _diag_warnings.append(f"{framework} has incomplete phase coverage for cells: {list(incomplete.index)}")

# Phase-label consistency: every framework present should only use the
# fixed PHASE_ORDER values (enforced by construction via the phase in
# PHASE_ORDER filter in each loader, but re-checked here defensively).
_bad_phases = set(phase_metric_long["phase_internal"].dropna().astype(str)) - set(PHASE_ORDER)
if _bad_phases:
    _diag_warnings.append(f"Unexpected phase_internal values outside PHASE_ORDER: {_bad_phases}")

# Every framework/cell should have all 3 phases (no silent partial rows).
_coverage = phase_metric_long.groupby(["framework", "cell_id"], observed=True)["phase_internal"].nunique()
_partial = _coverage[_coverage < len(PHASE_ORDER)]
if len(_partial):
    _diag_warnings.append(f"{len(_partial)} (framework, cell) combinations have <3 phases:\n{_partial}")

print(f"\n{len(_diag_warnings)} warning(s).")
for w in _diag_warnings:
    print("  [WARNING]", w)
if not _diag_warnings:
    print("  No issues found.")


In [ ]:
"""Code Cell 7 -- per-cell phase table builder and CSV export helper."""

from IPython.display import display


def _format_cell(rmse, mae) -> str:
    if pd.isna(rmse) or pd.isna(mae):
        return "\u2014"
    return f"{rmse:.2f} / {mae:.2f}"


def build_cell_phase_table(cell_id: str) -> pd.DataFrame:
    """Framework x phase RMSE/MAE table for one battery cell, reading only
    from the already-built `phase_metric_long` (never re-reading Drive per
    table)."""
    sub = phase_metric_long[phase_metric_long["cell_id"] == cell_id]

    rows = []
    for framework in FRAMEWORK_ROW_ORDER:
        fsub = sub[sub["framework"] == framework]
        row = {"Framework": framework}
        for phase in PHASE_ORDER:
            match = fsub[fsub["phase_internal"] == phase]
            col_name = PHASE_DISPLAY_MAP[phase]
            if len(match) == 0:
                row[col_name] = "\u2014"
            elif match["rmse"].nunique() > 1 or match["mae"].nunique() > 1:
                row[col_name] = "\u2014 (conflict)"
            else:
                row[col_name] = _format_cell(match["rmse"].iloc[0], match["mae"].iloc[0])
        rows.append(row)

    columns = ["Framework"] + [PHASE_DISPLAY_MAP[p] for p in PHASE_ORDER]
    return pd.DataFrame(rows, columns=columns)


def display_cell_phase_table(cell_id: str, table: pd.DataFrame) -> None:
    styler = (
        table.style
        .hide(axis="index")
        .set_properties(subset=["Framework"], **{"text-align": "left"})
        .set_properties(subset=[PHASE_DISPLAY_MAP[p] for p in PHASE_ORDER], **{"text-align": "center"})
        .set_caption(f"{cell_id} -- Phase-level RMSE / MAE (% SOC), Primary adapted only")
    )
    display(styler)
    print("Preview benchmark outputs. "
          "DARM, SFTNN(MED), and TATN use no target SOC labels during adaptation. "
          "DTNN and TATN values are means across five formal repeats.")
    table.to_csv(OUTPUT_DIR / f"{cell_id}_phase_metrics.csv", index=False)


In [ ]:
"""Code Cell 8 -- shared framework colors and per-cell / combined plotting
functions. Matplotlib only (no seaborn); academic, clean, publication-ready
styling; missing values are always gaps, never zero-filled or reflowed."""

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch

# Same 8-framework color mapping as notebook 38.
FRAMEWORK_COLORS = {
    "Vidal Fine-tuning":   "#2a78d6",  # blue
    "CNN-LSTM":            "#eb6834",  # orange
    "TCN":                 "#1baf7a",  # aqua
    "Transformer Encoder": "#eda100",  # yellow
    "DTNN":                "#e87ba4",  # magenta
    "TATN":                "#7f7f7f",  # grey
    "DARM":                "#008300",  # green
    "SFTNN (MED)":         "#4a3aa7",  # violet
}

ANNOTATE_BARS = False  # 8 frameworks x 3 phases = 24 bars per subplot; off by default

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.axisbelow": True,
    "grid.alpha": 0.25,
    "grid.linewidth": 0.6,
    "font.size": 10,
})


def _framework_label(framework: str) -> str:
    return framework


def _draw_phase_bars(ax, cell_id: str, metric_col: str):
    """Draw one cell's grouped bar chart (phases on x, frameworks as bars
    within each phase group) onto an existing Axes. Shared by both the
    single-cell and combined-figure paths so their bar geometry can never
    drift apart."""
    sub_cell = phase_metric_long[phase_metric_long["cell_id"] == cell_id]
    n_frameworks = len(FRAMEWORK_ROW_ORDER)
    n_phases = len(PHASE_ORDER)
    group_width = 0.82
    bar_width = group_width / n_frameworks

    for fi, framework in enumerate(FRAMEWORK_ROW_ORDER):
        fsub = sub_cell[sub_cell["framework"] == framework].set_index("phase_internal")
        xs, heights = [], []
        for pi, phase in enumerate(PHASE_ORDER):
            if phase not in fsub.index:
                continue
            value = fsub.loc[phase, metric_col]
            if pd.isna(value):
                continue
            x = pi - group_width / 2 + (fi + 0.5) * bar_width
            xs.append(x)
            heights.append(value)
        bars = ax.bar(
            xs, heights, width=bar_width * 0.92, color=FRAMEWORK_COLORS[framework],
            edgecolor="white", linewidth=0.6, label=_framework_label(framework), zorder=3,
        )
        if ANNOTATE_BARS:
            for b in bars:
                ax.annotate(f"{b.get_height():.1f}", (b.get_x() + b.get_width() / 2, b.get_height()),
                            xytext=(0, 2), textcoords="offset points", ha="center", va="bottom", fontsize=7)

    ax.set_xticks(range(n_phases))
    ax.set_xticklabels([PHASE_DISPLAY_MAP[p] for p in PHASE_ORDER])
    ax.set_ylim(bottom=0)
    ax.grid(True, axis="y", alpha=0.25, linewidth=0.6, zorder=0)


def plot_single_cell_phase_metric(cell_id: str, metric: str = "rmse"):
    """Single-cell grouped bar chart: x=phases (PHASE_ORDER), bars=frameworks
    (FRAMEWORK_ROW_ORDER), short cell_id-only title, its own external/top
    legend."""
    metric_col = "rmse" if metric == "rmse" else "mae"
    y_label = "RMSE (% SOC)" if metric == "rmse" else "MAE (% SOC)"

    fig, ax = plt.subplots(figsize=(7, 5.2))
    _draw_phase_bars(ax, cell_id, metric_col)
    ax.set_ylabel(y_label)
    ax.set_xlabel("Flight phase")
    ax.set_title(cell_id, fontsize=12, pad=54)

    handles = [Patch(facecolor=FRAMEWORK_COLORS[f], edgecolor="white", label=_framework_label(f))
               for f in FRAMEWORK_ROW_ORDER]
    ax.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=4, frameon=False, fontsize=8)

    fig.tight_layout()
    out_stem = OUTPUT_DIR / f"{cell_id}_phase_{metric}"
    fig.savefig(out_stem.with_suffix(".png"), dpi=300, bbox_inches="tight")
    if metric == "rmse":
        # Only RMSE PNGs are part of the requested per-cell PNG set this pass.
        pass
    return fig, ax


def plot_combined_phase_rmse_2x2x2x1():
    """Combined RMSE figure across all 7 cells, laid out 2+2+2+1 via
    GridSpec: 4 rows x 4 cols, each of the first 6 subplots spans 2 cols,
    and the 7th (VAH11) sits centered at gs[3, 1:3] -- same width as the
    paired subplots above it, never full-width. One shared legend only."""
    fig = plt.figure(figsize=(14, 15))
    gs = GridSpec(4, 4, figure=fig, top=0.90, bottom=0.08, hspace=0.55, wspace=0.35)

    slot_specs = [
        (0, slice(0, 2)), (0, slice(2, 4)),
        (1, slice(0, 2)), (1, slice(2, 4)),
        (2, slice(0, 2)), (2, slice(2, 4)),
        (3, slice(1, 3)),  # VAH11: centered, same width as the pairs above
    ]

    for cell_id, (row, col_slice) in zip(CELL_IDS, slot_specs):
        ax = fig.add_subplot(gs[row, col_slice])
        _draw_phase_bars(ax, cell_id, "rmse")
        ax.set_title(cell_id, fontsize=11)
        ax.set_xlabel("Flight phase")
        ax.set_ylabel("RMSE (% SOC)")

    handles = [Patch(facecolor=FRAMEWORK_COLORS[f], edgecolor="white", label=_framework_label(f))
               for f in FRAMEWORK_ROW_ORDER]
    fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0.985), ncol=4, frameon=False, fontsize=10)

    fig.suptitle("Phase-level RMSE by framework and target cell (Primary adapted only)", y=1.0, fontsize=13)

    fig.savefig(OUTPUT_DIR / "phase_rmse_all_cells_2x2x2x1.png", dpi=300, bbox_inches="tight")
    fig.savefig(OUTPUT_DIR / "phase_rmse_all_cells_2x2x2x1.pdf", bbox_inches="tight")
    return fig


In [ ]:
"""Code Cell -- VAH01 phase-level table and RMSE figure (independent
cell, per instruction: one code cell per battery cell, no shared for-loop)."""

table_vah01 = build_cell_phase_table("VAH01")
display_cell_phase_table("VAH01", table_vah01)
fig_vah01, ax_vah01 = plot_single_cell_phase_metric("VAH01", metric="rmse")
plt.show()


In [ ]:
"""Code Cell -- VAH17 phase-level table and RMSE figure (independent
cell, per instruction: one code cell per battery cell, no shared for-loop)."""

table_vah17 = build_cell_phase_table("VAH17")
display_cell_phase_table("VAH17", table_vah17)
fig_vah17, ax_vah17 = plot_single_cell_phase_metric("VAH17", metric="rmse")
plt.show()


In [ ]:
"""Code Cell -- VAH27 phase-level table and RMSE figure (independent
cell, per instruction: one code cell per battery cell, no shared for-loop)."""

table_vah27 = build_cell_phase_table("VAH27")
display_cell_phase_table("VAH27", table_vah27)
fig_vah27, ax_vah27 = plot_single_cell_phase_metric("VAH27", metric="rmse")
plt.show()


In [ ]:
"""Code Cell -- VAH30 phase-level table and RMSE figure (independent
cell, per instruction: one code cell per battery cell, no shared for-loop)."""

table_vah30 = build_cell_phase_table("VAH30")
display_cell_phase_table("VAH30", table_vah30)
fig_vah30, ax_vah30 = plot_single_cell_phase_metric("VAH30", metric="rmse")
plt.show()


In [ ]:
"""Code Cell -- VAH15 phase-level table and RMSE figure (independent
cell, per instruction: one code cell per battery cell, no shared for-loop)."""

table_vah15 = build_cell_phase_table("VAH15")
display_cell_phase_table("VAH15", table_vah15)
fig_vah15, ax_vah15 = plot_single_cell_phase_metric("VAH15", metric="rmse")
plt.show()


In [ ]:
"""Code Cell -- VAH12 phase-level table and RMSE figure (independent
cell, per instruction: one code cell per battery cell, no shared for-loop)."""

table_vah12 = build_cell_phase_table("VAH12")
display_cell_phase_table("VAH12", table_vah12)
fig_vah12, ax_vah12 = plot_single_cell_phase_metric("VAH12", metric="rmse")
plt.show()


In [ ]:
"""Code Cell -- VAH11 phase-level table and RMSE figure (independent
cell, per instruction: one code cell per battery cell, no shared for-loop)."""

table_vah11 = build_cell_phase_table("VAH11")
display_cell_phase_table("VAH11", table_vah11)
fig_vah11, ax_vah11 = plot_single_cell_phase_metric("VAH11", metric="rmse")
plt.show()


In [ ]:
"""Code Cell -- combined 2+2+2+1 phase-level RMSE figure across all 7 cells."""

fig_combined = plot_combined_phase_rmse_2x2x2x1()
plt.show()


In [ ]:
"""Code Cell -- combined machine-readable export."""

phase_metric_long.to_csv(OUTPUT_DIR / "all_phase_metrics_long.csv", index=False)
_all_phase_csv_path = OUTPUT_DIR / "all_phase_metrics_long.csv"
print(f"Wrote {_all_phase_csv_path} ({len(phase_metric_long)} rows)")
print(f"\nFiles written to {OUTPUT_DIR}/:")
for p in sorted(OUTPUT_DIR.glob("*")):
    if p.is_file():
        print(" ", p.name)


## Deferred (not in this pass)

Across-cell averaging / mean-median summaries (a possible future step) and
a combined MAE figure (the `metric="mae"` path of
`plot_single_cell_phase_metric` is pre-built but only invoked per-cell,
never assembled into a combined GridSpec figure) are both out of scope
this pass.